In [1]:
import pandas as pd
import numpy as np

In [2]:
df2 = pd.read_csv("https://raw.githubusercontent.com/erkansirin78/datasets/master/adult_preprocessed.csv")

In [3]:
df2.head()

,workclass,education,education_merged,marital_status,occupation,relationship,race,sex,native_country,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,output
0,State-gov,Bachelors,Under-Graduate,Never-married,Adm-clerical,Not-in-family,White,Male,United-States,39,77516.0,13.0,2174.0,0.0,40.0,<=50K
1,Self-emp-not-inc,Bachelors,Under-Graduate,Married-civ-spouse,Exec-managerial,Husband,White,Male,United-States,50,83311.0,13.0,0.0,0.0,13.0,<=50K
2,Private,HS-grad,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,United-States,38,215646.0,9.0,0.0,0.0,40.0,<=50K
3,Private,11th,High-School,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,United-States,53,234721.0,7.0,0.0,0.0,40.0,<=50K
4,Private,Bachelors,Under-Graduate,Married-civ-spouse,Prof-specialty,Wife,Black,Female,Cuba,28,338409.0,13.0,0.0,0.0,40.0,<=50K


In [4]:
df = df2[['workclass', 'education_merged', 'age','fnlwgt','education_num','capital_gain','capital_loss', 'hours_per_week','output']]

In [5]:
df.head()

,workclass,education_merged,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,output
0,State-gov,Under-Graduate,39,77516.0,13.0,2174.0,0.0,40.0,<=50K
1,Self-emp-not-inc,Under-Graduate,50,83311.0,13.0,0.0,0.0,13.0,<=50K
2,Private,HS-grad,38,215646.0,9.0,0.0,0.0,40.0,<=50K
3,Private,High-School,53,234721.0,7.0,0.0,0.0,40.0,<=50K
4,Private,Under-Graduate,28,338409.0,13.0,0.0,0.0,40.0,<=50K


In [6]:
# Feature matrix
X = df.iloc[:, :-1].values
print(X.shape)
print(X[:3])

(45207, 8)
[['State-gov' 'Under-Graduate' 39 77516.0 13.0 2174.0 0.0 40.0]
 ['Self-emp-not-inc' 'Under-Graduate' 50 83311.0 13.0 0.0 0.0 13.0]
 ['Private' 'HS-grad' 38 215646.0 9.0 0.0 0.0 40.0]]


In [7]:
# Output variable
y = df.iloc[:, -1]
print(y.shape)
print(y[:6])

(45207,)
0    <=50K
1    <=50K
2    <=50K
3    <=50K
4    <=50K
5    <=50K
Name: output, dtype: object


In [8]:
# split test train
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [10]:
# pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

In [11]:
pipeline = Pipeline([
    ('ct-ohe', ColumnTransformer(
        [('ct', 
          OneHotEncoder(handle_unknown='ignore', categories='auto'), 
          [0,1])], remainder='passthrough')
    ),
    ('scaler', StandardScaler(with_mean=False)),
    ('estimator', RandomForestClassifier(n_estimators=50))
])

In [12]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('ct-ohe',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ct',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  [0, 1])])),
                ('scaler', StandardScaler(with_mean=False)),
                ('estimator', RandomForestClassifier(n_estimators=50))])

In [13]:
ct = pipeline[0]

ct

ColumnTransformer(remainder='passthrough',
                  transformers=[('ct', OneHotEncoder(handle_unknown='ignore'),
                                 [0, 1])])

In [14]:
pipeline[-1]

RandomForestClassifier(n_estimators=50)

# Model evaluation

In [22]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [23]:
y_pred = pipeline.predict(X_test)

In [24]:
labelencoder_y = LabelEncoder()
y_pred = labelencoder_y.fit_transform(y_pred)

In [25]:
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [26]:
roc_auc_score(y_test, y_pred)

0.7045709169383351

# Raw prediction

In [27]:
single_pred_raw = np.array([['Self-emp-not-inc','Under-Graduate', 50, 83311.0, 13.0, 0.0, 0.0,13.0]])

In [28]:
single_pred_raw

array([['Self-emp-not-inc', 'Under-Graduate', '50', '83311.0', '13.0',
        '0.0', '0.0', '13.0']], dtype='<U32')

In [29]:
pipeline.predict(single_pred_raw)

array(['<=50K'], dtype=object)

# Save Model

In [30]:
import joblib

In [31]:
joblib.dump(pipeline, "pipeline_adult_random_forest.pkl")

['pipeline_adult_random_forest.pkl']

In [32]:
pipeline_loaded = joblib.load("pipeline_adult_random_forest.pkl")
print(type(pipeline_loaded))

<class 'sklearn.pipeline.Pipeline'>


# Upload Model to MinIO

In [20]:
#! mc cp pipeline_adult_random_forest.pkl mlops_minio/ml-models-bucket

In [33]:
import tempfile
import boto3
from botocore.config import Config
import logging
import joblib

In [34]:
ACCESS_KEY = 'root'
SECRET_KEY = 'root12345'

In [35]:
def get_s3_client():
    s3 = boto3.client('s3',
                      endpoint_url='http://minio:9000',
                      aws_access_key_id=ACCESS_KEY,
                      aws_secret_access_key=SECRET_KEY,
                      config=Config(signature_version='s3v4'))
    return s3


def get_s3_resource():
    s3_res = boto3.resource('s3',
                            endpoint_url='http://minio:9000',
                            aws_access_key_id=ACCESS_KEY,
                            aws_secret_access_key=SECRET_KEY,
                            config=Config(signature_version='s3v4'))
    return s3_res

In [36]:
def save_model_to_s3(model, bucket, key):
    ''' Store model as a buffer, then save buffer to s3'''
    s3_client = get_s3_client()
    try:
        with tempfile.TemporaryFile() as fp:
            joblib.dump(model, fp)
            fp.seek(0)
            s3_client.put_object(Body=fp.read(), Bucket=bucket, Key=key)
        logging.info(f'{key} saved to s3 bucket {bucket}')
    except Exception as e:
        raise logging.exception(e)

In [37]:
save_model_to_s3(pipeline, 'ml-models-bucket', 'pipeline_adult_random_forest.pkl')

# Read model from MinIO

In [38]:
def load_model_from_s3(bucket, key):
    ''' Read model from a s3 bucket'''
    s3_client = get_s3_client()
    try:
        with tempfile.TemporaryFile() as fp:
            s3_client.download_fileobj(Fileobj=fp, Bucket=bucket, Key=key)
            fp.seek(0)
            return joblib.load(fp)
    except Exception as e:
        raise logging.exception(e)

In [39]:
pipeline_loaded = load_model_from_s3('ml-models-bucket', 'pipeline_adult_random_forest.pkl')

In [40]:
pipeline_loaded.predict(single_pred_raw)

array(['<=50K'], dtype=object)